# ClinVar EDA

Raw parquet from `python src/ingest_clinvar.py`.


In [1]:
from pathlib import Path
import pandas as pd
import os
from dotenv import load_dotenv

In [2]:

load_dotenv()
project_root = Path(os.environ["PROJECT_ROOT"])
CLINVAR_PARQUET = project_root / "data/raw/clinvar_reliable_grch38.parquet"

In [3]:
df = pd.read_parquet(CLINVAR_PARQUET)
df.shape


(21870, 15)

In [4]:
df.columns

Index(['Type', 'Name', 'GeneSymbol', 'HGNC_ID', 'ClinicalSignificance',
       'LastEvaluated', 'PhenotypeIDS', 'PhenotypeList', 'Assembly',
       'Chromosome', 'Start', 'Stop', 'ReviewStatus', 'NumberSubmitters',
       'VariationID'],
      dtype='object')

In [5]:
df.head(1)


,Type,Name,GeneSymbol,HGNC_ID,ClinicalSignificance,LastEvaluated,PhenotypeIDS,PhenotypeList,Assembly,Chromosome,Start,Stop,ReviewStatus,NumberSubmitters,VariationID
0,single nucleotide variant,NM_000552.5(VWF):c.4883T>C (p.Ile1628Thr),VWF,HGNC:12726,Pathogenic,"Aug 13, 2024","MONDO:MONDO:0015628,MedGen:C1282968,Orphanet:1...",Von Willebrand disease type 2A|not provided|He...,GRCh38,12,6018535,6018535,reviewed by expert panel,16,284


In [6]:
df[df['VariationID']==54970]

,Type,Name,GeneSymbol,HGNC_ID,ClinicalSignificance,LastEvaluated,PhenotypeIDS,PhenotypeList,Assembly,Chromosome,Start,Stop,ReviewStatus,NumberSubmitters,VariationID
3995,single nucleotide variant,NM_007294.4(BRCA1):c.3708T>G (p.Asn1236Lys),BRCA1,HGNC:1100,Benign,"Jun 18, 2019","MONDO:MONDO:0003582,MeSH:D061325,MedGen:C06777...",Hereditary breast ovarian cancer syndrome|Brea...,GRCh38,17,43091823,43091823,reviewed by expert panel,37,54970


In [7]:
df['Type'].value_counts(dropna=False).head(20)


Type
single nucleotide variant    15306
Deletion                      3851
Duplication                   1418
Microsatellite                 517
Insertion                      476
Indel                          294
Inversion                        7
Variation                        1
Name: count, dtype: int64

In [8]:
# df['ReferenceAllele'].value_counts(dropna=False).head(20)  # all empty
# df['AlternateAllele'].value_counts(dropna=False).head(20)  # all empty


In [9]:
df['ReviewStatus'].value_counts(dropna=False)


ReviewStatus
reviewed by expert panel    21816
practice guideline             54
Name: count, dtype: int64

In [10]:
df['ClinicalSignificance'].value_counts(dropna=False)


ClinicalSignificance
Pathogenic                                                     9575
Uncertain significance                                         3811
Likely pathogenic                                              2788
Benign                                                         2769
Likely benign                                                  2760
drug response                                                   106
Pathogenic; drug response                                        39
Likely pathogenic; drug response                                 11
Uncertain significance; drug response                             3
Conflicting classifications of pathogenicity; drug response       1
not provided                                                      1
other                                                             1
Pathogenic/Likely pathogenic; drug response                       1
VUS-high                                                          1
Pathogenic/Likely pathogeni

In [11]:
df['PhenotypeList'].value_counts(dropna=False).head(20)


PhenotypeList
Breast-ovarian cancer, familial, susceptibility to, 2                                                                                                                                                    1115
Breast-ovarian cancer, familial, susceptibility to, 1                                                                                                                                                    1104
Hereditary thrombocytopenia and hematological cancer predisposition syndrome associated with RUNX1|Hereditary thrombocytopenia and hematologic cancer predisposition syndrome                             686
Monogenic diabetes                                                                                                                                                                                        423
Lynch syndrome                                                                                                                                                    

In [12]:
df_tmp = df[
    df["PhenotypeList"].isna() | (df["PhenotypeList"] == "not provided") | (df["PhenotypeList"] == "not specified")
]
df_tmp['ClinicalSignificance'].value_counts(dropna=False)


Series([], Name: count, dtype: int64)

In [13]:
df_tmp = df[df["ClinicalSignificance"] == "Uncertain significance"]
df_tmp['PhenotypeList'].value_counts(dropna=False)


PhenotypeList
Hereditary thrombocytopenia and hematological cancer predisposition syndrome associated with RUNX1|Hereditary thrombocytopenia and hematologic cancer predisposition syndrome                                                                                               351
Monogenic diabetes                                                                                                                                                                                                                                                          164
Hereditary thrombocytopenia and hematological cancer predisposition syndrome associated with RUNX1|Hereditary thrombocytopenia and hematologic cancer predisposition syndrome|Inborn genetic diseases                                                                       156
Open-angle glaucoma                                                                                                                                                       

## Scope characterization (indexed data)

Everything below describes the **cleaned/indexed** ClinVar subset (`data/processed/clinvar_rag.parquet`) — i.e. what the RAG can actually retrieve — and feeds `docs/scope.md`.

Topical coverage is **illustrative, not a closed list**: any concept appearing in the free-text fields (`Name`, `PhenotypeList`) is semantically retrievable.

In [14]:
import re

cvi = pd.read_parquet(project_root / "data/processed/clinvar_rag.parquet")
print("variants:", len(cvi), "| genes:", cvi["GeneSymbol"].astype(str).nunique())
print("assembly:", sorted(cvi["Assembly"].astype(str).unique()))
print("variant types:", sorted(cvi["Type"].astype(str).unique()))
print("chromosomes:", sorted(cvi["Chromosome"].astype(str).unique()))
print("\nreview status:")
print(cvi["ReviewStatus"].value_counts())
print("\nclinical significance:")
print(cvi["ClinicalSignificance"].value_counts())

variants: 11591 | genes: 178
assembly: ['GRCh38']
variant types: ['single nucleotide variant']
chromosomes: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '3', '4', '5', '6', '7', '9', 'MT', 'X']

review status:
ReviewStatus
reviewed by expert panel    11572
practice guideline             19
Name: count, dtype: int64

clinical significance:
ClinicalSignificance
Pathogenic                      4072
Likely benign                   2700
Benign                          2513
Likely pathogenic               2304
Pathogenic/Likely pathogenic       1
Benign/Likely benign               1
Name: count, dtype: int64


In [15]:
# Top phenotypes (illustrative topical coverage).
# Cleaned PhenotypeList joins phenotypes with ". " (from "|") and ";" for sub-phenotypes.
terms = []
for v in cvi["PhenotypeList"].dropna().astype(str):
    for part in re.split(r"\. |;", v):
        p = part.strip()
        if p and p.lower() != "not provided":
            terms.append(p)
pd.Series(terms).value_counts().head(25)

Hereditary cancer-predisposing syndrome                                                               3739
Hereditary breast ovarian cancer syndrome                                                             2406
Breast-ovarian cancer, familial, susceptibility to, 2                                                 2052
Breast-ovarian cancer, familial, susceptibility to, 1                                                 1728
Familial cancer of breast                                                                              800
Lynch syndrome                                                                                         795
Monogenic diabetes                                                                                     623
Hereditary nonpolyposis colorectal neoplasms                                                           617
Hereditary thrombocytopenia and hematologic cancer predisposition syndrome                             615
Inborn genetic diseases              

In [16]:
# Free-text reachability: a concept is in scope if it appears in retrievable text,
# even when it is not a gene symbol (e.g. "hemoglobin").
def clinvar_mentions(keyword):
    text = cvi[["Name", "PhenotypeList"]].astype(str).agg(" ".join, axis=1)
    return int(text.str.contains(keyword, case=False, na=False).sum())

for kw in ["hemoglobin", "cancer", "diabetes", "cardiac", "retina", "muscle", "mitochond"]:
    print(f"{kw:12s} {clinvar_mentions(kw):5d} variant record(s)")

hemoglobin      11 variant record(s)
cancer        5712 variant record(s)


diabetes       632 variant record(s)


cardiac         11 variant record(s)
retina         183 variant record(s)


muscle           8 variant record(s)


mitochond      129 variant record(s)
